<a href="https://colab.research.google.com/github/Jesse-Goldie/MMA_3001_Notes/blob/Jesse/Jesse-Practical/MMA3001-Uncommented-Code-main/Commenting-On-Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from PIL import Image
import os
from glob import glob

def average_images(input_dir: str, output_path: str = "averaged.png"):
    """
    Averages a series of PNG images in a directory and saves the result.

    This function reads all PNG files from the specified input directory,
    calculates the element-wise average of their pixel values, and saves
    the resulting averaged image to the specified output path.

    Parameters
    ----------
    input_dir : str
        The path to the directory containing the PNG images.
    output_path : str, optional
        The path where the averaged image will be saved. Defaults to "averaged.png".

    Raises
    ------
    ValueError
        If no PNG images are found in the specified directory.
    """
    # Get a sorted list of all PNG files in the input directory
    files = sorted(glob(os.path.join(input_dir, "*.png")))

    # Raise an error if no PNG files are found
    if not files:
        raise ValueError("No PNG images found in directory.")

    # Open the first image to get its dimensions and initialize the accumulator
    first = np.array(Image.open(files[0]), dtype=np.float64)
    accumulator = np.zeros_like(first)

    # Iterate through each image file, open it, convert to NumPy array, and add to accumulator
    for f in files:
        accumulator += np.array(Image.open(f), dtype=np.float64)

    # Calculate the average by dividing the accumulator by the total number of images
    # Convert the result back to unsigned 8-bit integer format (0-255)
    averaged = (accumulator / len(files)).astype(np.uint8)

    # Create a PIL Image object from the averaged NumPy array
    out_img = Image.fromarray(averaged)

    # Save the averaged image to the specified output path
    out_img.save(output_path)
    print(f"Averaged image saved to {output_path}")


In [ ]:
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import os

def generate_noisy_images(
    output_dir: str,
    n_images: int = 20,
    width: int = 800,
    height: int = 400,
    noise_level: float = 0.25
):
    """
    Generates a series of noisy images with a central text.

    Each image will contain the text "MMA3001" centered, with random noise
    applied to it. The images are saved as PNG files in the specified output directory.

    Parameters
    ----------
    output_dir : str
        The directory where the generated noisy images will be saved.
    n_images : int, optional
        The number of noisy images to generate. Defaults to 20.
    width : int, optional
        The width of the generated images in pixels. Defaults to 800.
    height : int, optional
        The height of the generated images in pixels. Defaults to 400.
    noise_level : float, optional
        The intensity of the noise to be added to the images. Higher values
        result in more visible noise. Defaults to 0.25.

    Returns
    -------
    None
        The function saves images to disk and prints a confirmation message.

    """
    # Create the output directory if it does not exist
    os.makedirs(output_dir, exist_ok=True)

    # Attempt to load Arial font; fall back to default if not found
    try:
        font = ImageFont.truetype("arial.ttf", 80)
    except:
        font = ImageFont.load_default()

    # Loop to generate each image
    for i in range(n_images):
        # Create a new white RGB image
        img = Image.new("RGB", (width, height), color="white")
        draw = ImageDraw.Draw(img)

        # Define the text to be drawn
        text = "MMA3001"

        # Calculate text bounding box to center the text
        bbox = draw.textbbox((0, 0), text, font=font)
        text_w = bbox[2] - bbox[0]
        text_h = bbox[3] - bbox[1]
        pos = ((width - text_w) // 2, (height - text_h) // 2)

        # Draw the text in black on the image
        draw.text(pos, text, fill="black", font=font)

        # Generate Gaussian noise based on image dimensions and noise level
        noise = np.random.randn(height, width, 3) * 255 * noise_level

        # Add noise to the image, clip values to [0, 255], and convert to uint8
        noisy = np.clip(np.array(img) + noise, 0, 255).astype(np.uint8)

        # Convert the NumPy array back to a PIL Image
        noisy_img = Image.fromarray(noisy)

        # Save the noisy image to the output directory
        noisy_img.save(os.path.join(output_dir, f"noisy_{i:03d}.png"))

    # Print confirmation after generating all images
    print(f"Generated {n_images} noisy images in {output_dir}")
